In [7]:
import torch
import cv2

In [8]:
# Load the YOLOv7 model
model = torch.hub.load('WongKinYiu/yolov7', 'custom', 'yolov7/runs/train/exp4/weights/last.pt')
model.eval()

Using cache found in /home/tanawin/.cache/torch/hub/WongKinYiu_yolov7_main
/home/tanawin/.cache/torch/hub/WongKinYiu_yolov7_main/hubconf.py:68: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experim

Adding autoShape... 


autoShape(
  (model): Model(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU()
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU()
      )
      (2): Conv(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU()
      )
      (3): Conv(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU()
      )
 

In [9]:

def analyze(inputPath, outputPath):
    print("executing... ", inputPath)
    # vdo reader
    cap = cv2.VideoCapture(inputPath)
    # vdo writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    out = cv2.VideoWriter(outputPath, fourcc, fps, (frame_width, frame_height))


    # Loop through each frame of the video
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert frame to RGB format
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Perform detection
        results = model(img)

        # Extract detection data
        detections = results.pandas().xyxy[0]  # Bounding boxes with confidence and class

        # Draw bounding boxes and labels
        for _, row in detections.iterrows():
            xmin, ymin, xmax, ymax = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
            label = row['name']  # class name
            confidence = row['confidence']

            # Draw bounding box and label
            cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
            cv2.putText(frame, f"{label} {confidence:.2f}", (xmin, ymin - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        #display_frame = cv2.resize(frame, (display_width, display_height))
        # Display the frame
        out.write(frame)
    cap.release()
    out.release()
    print("finish")
    #cv2.destroyAllWindows()


        #cv2.imshow("YOLOv7 Chess Detection", frame)
        # if cv2.waitKey(1) & 0xFF == ord('q'):
        #     break

In [10]:
fileNames = [
    "2_Move_rotate_student.mp4",
    "4_Move_studet.mp4",
    "6_Move_student.mp4",
    "8_Move_student.mp4"

]

for name in fileNames:
    analyze("vdo/" + name, "out_vdo/" + name)



executing...  vdo/2_Move_rotate_student.mp4
finish
executing...  vdo/4_Move_studet.mp4
finish
executing...  vdo/6_Move_student.mp4
finish
executing...  vdo/8_Move_student.mp4
finish
